# Conference Fig. 3 — projected energy and DiffCSP generation on simulated PCM arrays

**Claim.** DiffCSP, 100 % crossbar MACs, is projected at ≈98× less energy than a measured GPU (ChargE3Net capped by Amdahl's law at ≈1.9×). With the noise-budget sampler, 96–99.5 % of DiffCSP's network evaluations run on the array and the generated structures keep the digital accuracy as the read noise grows (RMSE), where naive sampling degrades; the layered GST structures form during the analog part of the trajectory; with the last 100 steps digital, analog-generated (GeTe)ₘ(Sb₂Te₃)ₙ reach the digital mixing energies, including compositions not in MP.

- **a** Energy relative to the measured A800 GPU (GPU = measured; PCM / hybrid = NeuroSim projection, stated in the caption, not in the legend): DiffCSP per structure (GPU batch 500; whisker = 2 columns per weight), ChargE3Net per 10⁶ grid points (hybrid = digital part on GPU + crossbar part on PCM). Third bar of each group (right axis): breakdown of the projected PCM crossbar energy (NeuroSim), with TOPS/W — `results/P1_neurosim/nominal_22nm/{compare,metrics}.json`.
- **b** 1000 MP-20 test structures: digital, and naive vs noise-budget sampling at read noise 1 / 3 / 5 % of G (dual slice, 3 chips, mean ± s.d.): match rate (top) and RMSE (bottom); dashed = digital; number in the budget bar = share of evaluations on the array — `results/T3_diffcsp_digital/eval1000_seed42`, `results/G1/eval1000_dual_chip*`, `results/G3/eval1000_budget_dual_chip*`, `results/G4/eval1000_{naive,budget}_dual_read{3,5}pct_chip*`.
- **c** Noise budget (chip 0): injected noise (solid) vs read-noise contribution (dashed) per sub-step; shaded = digital sub-steps. Top: one Ge₂Sb₂Te₅ sample at t = 1000 / 200 / 50 / 15 / 0 (evenly spaced; label blue = array, grey = digital; dotted line to its t) — curves `results/G3/eval1000_budget_dual_chip0/budget_curves.csv`; frames `results/S1/trajectory/Ge2Sb2Te5_chip0_seed0/traj.npz` (each frame in its own lattice rescaled to the final volume, final viewing frame).
- **d** Lowest MACE-MP-0 mixing energy per (GeTe)ₘ(Sb₂Te₃)ₙ composition, digital vs **analog = noise-budget sampling with the last 100 of 1000 steps digital** (3 chips: mean, bar = range); open squares = lowest-energy MP polymorph per composition (MACE-relaxed); E_mix = 0 is defined by the lowest MP GeTe and Sb₂Te₃; the end members themselves were also generated (seed 2, 8 samples). High-GeTe compositions (seed 3) exist in the data but are not drawn. Top: the lowest-energy analog structure of four compositions (relaxed), stacking axis horizontal, one period, same Å scale; shaded = van der Waals gaps — `results/showcase_test/summary_{digital,analog_chip*_tail100}_seed{0,1}.json`, `results/S1/gallery_tail/`.

**Differs from the nominal sampler (G3)**: in d the last 100 steps run digitally (90 % of evaluations on the array, stated in the caption); b and c are nominal.
Caveats: PCM energies are NeuroSim projections (22 nm), never measurements. d energies are MACE-MP-0 (known compositions seed 0, 8 samples; the three compositions not in MP seed 1, 32 samples; digital pooled over both seeds); GST compounds are in MP-20 training data (qualitative showcase only).


In [ ]:
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from style import *
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle
from pymatgen.core import Structure
W = W_CONF          # 会议版画布宽度 178 mm
SHOW = RESULTS / 'showcase_test'
GAL = RESULTS / 'S1' / 'gallery_tail'      # 模拟 = 噪声预算 + 末段 100 步数字
TRAJ = RESULTS / 'S1' / 'trajectory' / 'Ge2Sb2Te5_chip0_seed0'
CHIPS = (0, 1, 2)
NS = RESULTS / 'P1_neurosim' / 'nominal_22nm'
CMP = json.loads((NS / 'compare.json').read_text())
NSM = json.loads((NS / 'metrics.json').read_text())

def runs(d):
    # metrics of one run directory pattern (3 chips or the digital run)
    return [json.loads(p.read_text())['runs'][0] for p in sorted(RESULTS.glob(d + '/metrics.json'))]

def mix_best():
    # lowest mixing energy (meV/atom) per (mode, chip, composition); mode: digital / tail (= analog in the figure)
    rows = []
    for f in SHOW.glob('summary_*.json'):
        d = json.loads(f.read_text()); gen = d['gen']
        mode = 'digital' if gen.startswith('digital') else ('tail' if 'tail' in gen else 'analog')
        m = re.search(r'chip(\d)', gen); chip = int(m.group(1)) if m else -1
        rows += [dict(mode=mode, chip=chip, composition=r['composition'], E=r['Emix_gen_min_meV'])
                 for r in d['rows'] if r['group'] in ('gst_series', 'gst_end') and 'Emix_gen_min_meV' in r]
    return pd.DataFrame(rows).groupby(['mode', 'chip', 'composition']).E.min().reset_index()

# ---- structure drawing (b, c): view along an in-plane lattice vector, one stacking period ----
RADIUS = {'Ge': 0.55, 'Sb': 0.62, 'Te': 0.62}   # display radii (Å)
BOND_MAX = 3.25                                 # Å, Ge/Sb–Te bonds
GAP_MIN = 2.4                                   # Å, plane spacing drawn as a van der Waals gap
NI = 5                                          # lattice images per direction

def view_frame(L, frac):
    # Viewing frame from a reference cell: in-plane = two shortest rows, window bottom in the widest plane gap.
    ia, ib, ic = np.argsort(np.linalg.norm(L, axis=1))
    a, b, c = L[ia], L[ib], L[ic]
    ah = a / np.linalg.norm(a); n = np.cross(a, b); n /= np.linalg.norm(n); u = np.cross(n, ah)
    P = abs(c @ n)
    h = np.sort(((frac % 1) @ L @ n) % P)
    g = np.diff(np.r_[h, h[0] + P]); k = int(np.argmax(g))
    return dict(ah=ah, n=n, u=u, P=P, depth=np.linalg.norm(a), h0=h[k] + g[k] / 2)

def window_atoms(frac, L, species, vf, win):
    k = np.arange(-NI, NI + 1)
    sh = np.stack(np.meshgrid(k, k, k, indexing='ij'), -1).reshape(-1, 3)
    cart = ((frac[None] + sh[:, None]) @ L).reshape(-1, 3)
    sp = np.tile(np.asarray(species), len(sh))
    d, h, x = cart @ vf['ah'], cart @ vf['n'] - vf['h0'], cart @ vf['u']
    inside = (d >= 0) & (d < vf['depth']) & (h >= 0) & (h < vf['P'])
    x0 = np.median(x[inside])
    keep = inside & (np.abs(x - x0) < win / 2)
    return x[keep] - x0 + win / 2, h[keep], d[keep], cart[keep], sp[keep]

def draw_structure(ax, frac, L, species, vf, win, colors, edge, bond_c, gap_c, gaps=True, lw_atom=0.3, lw_bond=0.4,
                   horizontal=False):
    # horizontal=True: stacking axis along x (layers as vertical columns)
    x, h, d, cart, sp = window_atoms(frac, L, species, vf, win)
    X, Y = (h, x) if horizontal else (x, h)
    if gaps:
        hs = np.unique(np.round(np.sort(h), 1)); edges = np.r_[0, hs, vf['P']]
        for lo, hi in zip(edges[:-1], edges[1:]):
            full = lo > 0 and hi < vf['P']
            if (full and hi - lo > GAP_MIN) or (not full and hi - lo > GAP_MIN / 2):
                r = (lo, -0.8, hi - lo, win + 1.6) if horizontal else (-0.8, lo, win + 1.6, hi - lo)
                ax.add_patch(Rectangle(r[:2], r[2], r[3], fc=gap_c, ec='none', zorder=0))
    dist = np.linalg.norm(cart[:, None] - cart[None], axis=-1)
    for i, j in zip(*np.where(np.triu((dist > 0.5) & (dist < BOND_MAX)))):
        ax.plot([X[i], X[j]], [Y[i], Y[j]], color=bond_c, lw=lw_bond, zorder=1, solid_capstyle='round')
    for i in np.argsort(-d):
        ax.add_patch(Circle((X[i], Y[i]), RADIUS[str(sp[i])], fc=colors[str(sp[i])], ec=edge, lw=lw_atom, zorder=2))
    ax.set_aspect('equal'); ax.axis('off')

def comp_tex(comp):
    return ''.join(f'{el}$_{{{n}}}$' if n else el for el, n in re.findall(r'([A-Z][a-z]?)(\d*)', comp))


In [ ]:
# 字体设置 / fonts —— 全文唯一来源（style.PAPER_FONTS）
FS = use_paper_fonts()
FS_TITLE, FS_LABEL, FS_TICK = FS['title'], FS['label'], FS['tick']
FS_LEGEND, FS_ANNOT, FS_PANEL = FS['legend'], FS['annot'], FS['panel']
FS_CBLABEL, FS_CBTICK = FS['cbar_label'], FS['cbar_tick']


In [ ]:
# layout: one row of 4 panels, canvas W_CONF = 7.008 in; all boxes in inches (x0, y0, w, h)
# Derivation: bottom 0.36 (x labels), top 0.08, content height 1.45 -> H = 1.89 in (48 mm)
#   left 0.40 | a 1.04 (energy, right share axis; legend 0.42 above) | 0.72 (a's right axis + b's y label) | b 1.035
#   | 0.40 | c 1.25 (curves 0.95 + 0.04 + frames 0.46, evenly spaced) | 0.44 | d 1.58 (structures 2 x 2) -> 6.91 in
OUT = FIG_OUT / 'conf_fig3'
PANEL_OUT = OUT / 'panels'
PREVIEW_SCALE = 0.5
mpl.rcParams['savefig.transparent'] = True   # 导出背景透明（整图与单面板）
H = 1.89
BOT, CH = 0.36, 1.45
X_A, W_A = 0.40, 1.04
X_B, W_B = X_A + W_A + 0.72, 1.035
X_C, W_C = X_B + W_B + 0.40, 1.25
X_D, W_D = X_C + W_C + 0.44, 1.58
C_CURVE, C_FRAME_H = 0.95, 0.46
A_LEG = 0.42                 # a: legend band above the axes
D_TOP = 0.56                 # d: structure block height above the scatter

def make_layout():
    fig = plt.figure(figsize=(W, H))
    return fig, [(X_A, BOT, W_A, CH), (X_B, BOT, W_B, CH), (X_C, BOT, W_C, CH), (X_D, BOT, W_D, CH)]   # a-d

def add_ax(fig, x0, y0, w, h):
    return fig.add_axes([x0 / W, y0 / H, w / W, h / H])

def panel_run(i, letter, draw):
    fig, slots = make_layout()
    draw(fig, slots['abcd'.index(letter)])
    group = {'axes': list(fig.axes), 'texts': list(fig.texts)}
    boxes = save_matched_panels(fig, {letter: group}, PANEL_OUT); plt.close(fig)
    display(NotebookImage(filename=str(PANEL_OUT / f'{letter}.png'), width=int(boxes[letter].width * 600 * PREVIEW_SCALE)))


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Ca_GPU = '#7E7E7E'    # GPU 实测（归一化为 1）
Ca_PCM = '#4DBBD5'    # PCM（估算）；与构成柱的 ADC 深蓝区分
Ca_HYB = '#A8DCEB'    # 混合方案（数字部分仍在 GPU 上）
Ca_TXT = '#1F7A91'    # 倍数标注（PCM 色系）
Ca_PART = {'ADC': '#3C5488', 'Accum.': '#00A087', 'Periph.': '#F39B7F', 'Interc.': '#8491B4',
           'Buffer': '#91D1C2', 'Leakage': '#BDBDBD'}   # PCM 能耗构成（与原 Fig. 4e 一致）
Ca_SPLIT_EDGE = '#000000'
Ca_IN = '#FFFFFF'     # 构成柱内文字（TOPS/W）
# ----------------------------------------------------------------------
A_W = 0.32                       # bar width
A_GAP = 0.06                     # gap before the breakdown bar
A_X = {'DiffCSP': 0.0, 'ChargE3Net': 1.22}   # group start; bars: GPU, PCM, breakdown

def draw_a(fig, box):
    x0, y0, w, h = box
    ax = add_ax(fig, x0, y0, w, h - A_LEG)
    dn = CMP['diffcsp']['nominal']['batches']['500']; d2 = CMP['diffcsp']['cols2']['batches']['500']
    cn = CMP['charge3net']['nominal']; cb = cn['batches']['5000']
    ratio = {'DiffCSP': dn['energy_ratio_gpu_over_analog'], 'ChargE3Net': cb['energy_ratio_gpu_over_hybrid']}
    r_d2 = d2['energy_ratio_gpu_over_analog']
    for m, xs in A_X.items():
        ax.bar(xs, 1, A_W, color=Ca_GPU, label='GPU' if m == 'DiffCSP' else None)
        c = Ca_PCM if m == 'DiffCSP' else Ca_HYB
        ax.bar(xs + A_W, 1 / ratio[m], A_W, color=c, label='PCM' if m == 'DiffCSP' else 'hybrid')
        r = ratio[m]
        ax.text(xs + A_W, 1.25 / r, f'{r:.0f}×' if r > 10 else f'{r:.1f}×', ha='center', va='bottom',
                fontsize=FS_ANNOT - 1, color=Ca_TXT)
    ax.errorbar(A_X['DiffCSP'] + A_W, 1 / ratio['DiffCSP'], yerr=[[1 / ratio['DiffCSP'] - 1 / r_d2], [0]],
                color='k', lw=0.6, capsize=1.2)
    ax.set_yscale('log'); ax.set_ylim(3e-3, 4)
    ax.set_ylabel('Energy (GPU = 1)')
    # breakdown of the projected PCM (crossbar) energy, third bar of each group, right axis 0–100 %
    ax2 = ax.twinx()
    for m, key in (('DiffCSP', 'diffcsp/nominal'), ('ChargE3Net', 'charge3net/nominal')):
        c = NSM['runs'][key]['chip']
        parts = {'ADC': c['ADC_dynamic_pJ'], 'Accum.': c['accum_dynamic_pJ'],
                 'Periph.': c['other_dynamic_pJ'] - c['ic_dynamic_pJ'] - c['buffer_dynamic_pJ'],
                 'Interc.': c['ic_dynamic_pJ'], 'Buffer': c['buffer_dynamic_pJ'], 'Leakage': c['leakage_pJ']}
        tot = sum(max(v, 0) for v in parts.values()); yb = 0; xb = A_X[m] + 2 * A_W + A_GAP
        for name, v in parts.items():
            f = 100 * max(v, 0) / tot
            ax2.bar(xb, f, A_W, bottom=yb, color=Ca_PART[name], lw=0, label=name if m == 'DiffCSP' else None)
            yb += f
        ax2.bar(xb, 100, A_W, fill=False, ec=Ca_SPLIT_EDGE, lw=0.4)
        ax2.text(xb, 3, f'{c["tops_per_w"]:.2f} TOPS/W', rotation=90, ha='center', va='bottom',
                 fontsize=FS_ANNOT - 2, color=Ca_IN)
    ax2.set_ylim(0, 100); ax2.set_yticks([0, 50, 100]); ax2.set_ylabel('PCM share (%)', labelpad=1)
    ax2.spines['right'].set_visible(True)
    ax.set_xticks([A_X[m] + A_W + A_GAP / 2 for m in A_X], list(A_X))
    ax.set_xlim(-A_W / 2 - 0.08, A_X['ChargE3Net'] + 2 * A_W + A_GAP + A_W / 2 + 0.08)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='lower left', bbox_to_anchor=(-0.02, 1.0), ncol=3, fontsize=FS_LEGEND - 1,
              handlelength=0.8, handletextpad=0.3, columnspacing=0.5, labelspacing=0.2, borderaxespad=0.1)
    return ax
panel_run(0, 'a', draw_a)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cb_NAIVE = '#E64B35'    # 朴素采样
Cb_BUDGET = '#3C5488'   # 噪声预算
Cb_DIG = '#7E7E7E'      # 数字基线（柱 + 贯穿虚线）
Cb_ERR = '#000000'
Cb_IN = '#FFFFFF'       # 柱内文字（阵列占比）
Cb_HATCH = '#000000'    # 朴素采样柱的斜线阴影
# ----------------------------------------------------------------------
B_RUNS = {'naive': {1: 'G1/eval1000_dual_chip*', 3: 'G4/eval1000_naive_dual_read3pct_chip*',
                    5: 'G4/eval1000_naive_dual_read5pct_chip*'},
          'budget': {1: 'G3/eval1000_budget_dual_chip*', 3: 'G4/eval1000_budget_dual_read3pct_chip*',
                     5: 'G4/eval1000_budget_dual_read5pct_chip*'}}
B_BW = 0.36   # bar width (group spacing 1)
mpl.rcParams['hatch.linewidth'] = 0.4   # 斜线阴影线宽（本图仅 b 用到）
B_XDIG = -0.9 # 数字基线柱的位置（读噪声组左侧）

def draw_b(fig, box):
    x0, y0, w, h = box
    hh = (h - 0.14) / 2
    ax1 = add_ax(fig, x0, y0 + hh + 0.14, w, hh)
    ax2 = add_ax(fig, x0, y0, w, hh)
    dig = runs('T3_diffcsp_digital/eval1000_seed42')[0]
    for ax, key, scale in ((ax1, 'match_rate', 100), (ax2, 'rmse', 1)):
        ax.bar(B_XDIG, dig[key] * scale, width=B_BW, color=Cb_DIG, label='digital' if ax is ax1 else None)
        ax.axhline(dig[key] * scale, color=Cb_DIG, lw=0.5, ls='--', zorder=3)
    for name, c, dx in (('naive', Cb_NAIVE, -B_BW / 2), ('budget', Cb_BUDGET, B_BW / 2)):
        for g, (r, pat) in enumerate(B_RUNS[name].items()):
            rr = runs(pat)
            mr = np.array([q['match_rate'] * 100 for q in rr]); rm = np.array([q['rmse'] for q in rr])
            kw = dict(color=c, width=B_BW, error_kw=dict(ecolor=Cb_ERR, lw=0.5, capsize=1.2))
            if name == 'naive':
                kw.update(hatch='//////', edgecolor=Cb_HATCH, lw=0)   # 斜线阴影（线宽见 rcParams['hatch.linewidth']）
            ax1.bar(g + dx, mr.mean(), yerr=mr.std(), label=name if g == 0 else None, **kw)
            ax2.bar(g + dx, rm.mean(), yerr=rm.std(), **kw)
            if name == 'budget':
                f = np.mean([q['analog_fraction'] for q in rr])
                ax1.text(g + dx, 3, f'{100 * f:.1f}%', rotation=90, ha='center', va='bottom',
                         fontsize=FS_ANNOT - 2, color=Cb_IN)
    ax1.set_ylim(0, 88); ax1.set_yticks([0, 30, 60]); ax1.set_ylabel('Match (%)')
    ax2.set_ylim(0, 0.105); ax2.set_yticks([0, 0.05, 0.1]); ax2.set_ylabel('RMSE')
    for ax in (ax1, ax2):
        ax.set_xlim(B_XDIG - 0.45, 2.6); ax.set_xticks([B_XDIG, 0, 1, 2])
    ax1.set_xticklabels([]); ax2.set_xticklabels(['dig.', '1', '3', '5'])
    ax2.set_xlabel('Read noise (% of G)')
    ax1.legend(loc='upper center', fontsize=FS_LEGEND - 1, handlelength=0.8, handletextpad=0.3, ncol=3,
               columnspacing=0.5, borderaxespad=0.05)
    return ax1
panel_run(1, 'b', draw_b)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cc_SUB = {'corr': '#00A087', 'pred': '#3C5488', 'latt': '#E64B35'}   # corrector x / predictor x / predictor L
Cc_DIG = '#EDEDED'      # 切回数字的时间步
Cc_STYLE = '#000000'    # 线型图例
Cc_ATOM = {'Ge': '#E64B35', 'Sb': '#3C5488', 'Te': '#BDBDBD'}   # 与图 0 一致
Cc_ATOM_EDGE = '#000000'
Cc_BOND = '#8C8C8C'
Cc_GAP = '#EEF1F7'
Cc_T_ANALOG = '#3C5488'   # 帧标签：阵列上
Cc_T_DIGITAL = '#9E9E9E'  # 帧标签：数字
Cc_LINK = '#B0B0B0'
# ----------------------------------------------------------------------
C_SAMPLE = 0
C_FRAMES = (1000, 200, 50, 15, 0)
C_WIN = 5.0          # Å, lateral window of a frame
C_HSTRUCT = 19.5     # Å, frame height (unrelaxed Ge2Sb2Te5 period 18.6 Å)

def draw_c(fig, box):
    x0, y0, w, h = box
    ax = add_ax(fig, x0, y0, w, C_CURVE)
    d = pd.read_csv(RESULTS / 'G3/eval1000_budget_dual_chip0/budget_curves.csv').sort_values('t')
    d = d[d.t > 1]
    for key, std, s in (('corr', 'std_x_corr', 's_x_corr'), ('pred', 'std_x_pred', 's_x_pred'),
                        ('latt', 'std_l_pred', 's_l_pred')):
        ax.plot(d.t, d[std], '-', color=Cc_SUB[key], lw=0.9)
        ax.plot(d.t, d[s], '--', color=Cc_SUB[key], lw=0.7)
    dig = d[~(d.analog_corr & d.analog_pred)]
    t_dig = int(dig.t.max()) if len(dig) else 0
    if len(dig):
        ax.axvspan(0.5, t_dig + 0.5, color=Cc_DIG, zorder=0, lw=0)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(1000, 1.5); ax.set_ylim(1e-7, 3); ax.set_yticks([1e-6, 1e-4, 1e-2, 1])
    ax.set_xlabel('Diffusion step t'); ax.set_ylabel('Noise std')
    l1 = ax.legend([Line2D([], [], color=c, lw=0.9) for c in Cc_SUB.values()], ['corr., x', 'pred., x', 'pred., L'],
                   loc='lower left', fontsize=FS_LEGEND - 1, handlelength=1.0, borderaxespad=0.1, ncol=3,
                   columnspacing=0.4, handletextpad=0.3)
    ax.add_artist(l1)
    ax.legend([Line2D([], [], color=Cc_STYLE, lw=0.9), Line2D([], [], color=Cc_STYLE, lw=0.7, ls='--')],
              ['injected', 'read noise'], loc='upper right', fontsize=FS_LEGEND - 1, handlelength=1.2,
              borderaxespad=0.1, ncol=2, columnspacing=0.6)
    # frames of one sample above the axis, evenly spaced; dotted line to each frame's t
    z = np.load(TRAJ / 'traj.npz'); ts = list(z['t']); sp = [str(e) for e in z['species']]
    fr, lat = z['frac'][:, C_SAMPLE], z['lattice'][:, C_SAMPLE]
    L0 = lat[ts.index(0)]; vf = view_frame(L0, fr[ts.index(0)]); V0 = abs(np.linalg.det(L0))
    fh = C_FRAME_H                            # frame box height (in); label to the right, rotated
    fw = (C_WIN + 1.6) * fh / C_HSTRUCT
    lx = lambda t: (np.log10(1000) - np.log10(max(t, 1.5))) / (np.log10(1000) - np.log10(1.5)) * w
    step = (w - fw - 0.06) / (len(C_FRAMES) - 1)   # frames evenly spaced over the axis width
    for i, t in enumerate(C_FRAMES):
        cx = fw / 2 + i * step
        fa = add_ax(fig, x0 + cx - fw / 2, y0 + C_CURVE + 0.04, fw, fh)
        L = lat[ts.index(t)]; L = L * (V0 / abs(np.linalg.det(L))) ** (1 / 3)
        draw_structure(fa, fr[ts.index(t)] % 1, L, sp, vf, C_WIN, Cc_ATOM, Cc_ATOM_EDGE, Cc_BOND, Cc_GAP,
                       gaps=(t == 0), lw_atom=0.2, lw_bond=0.25)
        fa.set_xlim(-0.8, C_WIN + 0.8); fa.set_ylim(-0.6, C_HSTRUCT - 0.6)
        fa.text(1.05, 0.5, f't = {t}', transform=fa.transAxes, ha='left', va='center', rotation=90,
                fontsize=FS_ANNOT - 2, color=Cc_T_ANALOG if t > t_dig else Cc_T_DIGITAL)
        fig.add_artist(Line2D([(x0 + cx) / W, (x0 + lx(t)) / W], [(y0 + C_CURVE + 0.04) / H, (y0 + C_CURVE) / H],
                              color=Cc_LINK, lw=0.4, ls=':'))
    return ax
panel_run(2, 'c', draw_c)


In [ ]:
# ---------------- 本面板配色（独立可改，不影响其它面板） ----------------
Cd_DIG = '#7E7E7E'      # 数字生成，每组分最低
Cd_ANA = '#3C5488'      # 模拟 = 噪声预算 + 末段 100 步数字（3 芯片：均值，竖线 = 范围）
Cd_END = '#7E7E7E'      # 端点 GeTe / Sb2Te3（参考，E_mix = 0）
Cd_MP = '#000000'       # MP 已知多型体（空心方块）
Cd_ZERO = '#BDBDBD'
Cd_ATOM = {'Ge': '#E64B35', 'Sb': '#3C5488', 'Te': '#BDBDBD'}
Cd_ATOM_EDGE = '#000000'
Cd_BOND = '#8C8C8C'
Cd_GAP = '#EEF1F7'
# ----------------------------------------------------------------------
D_SHOW = ('GeSb4Te7', 'GeSb2Te4', 'Ge2Sb2Te5', 'Ge4Sb2Te7')   # 上方展示的模拟结构：两行，按 GeTe 比例排序
D_COMPS = ('Sb2Te3', 'GeSb6Te10', 'GeSb4Te7', 'GeSb2Te4', 'Ge3Sb4Te9', 'Ge2Sb2Te5', 'Ge3Sb2Te6', 'Ge4Sb2Te7',
           'GeTe')   # 画入散点的组分（不含高 x 侧 Ge5Sb2Te8 / Ge6Sb2Te9 / Ge7Sb2Te10）
D_WIN = 3.7          # Å, lateral window (≈ one in-plane period)
D_LMAX = 27.0        # Å, longest stacking period drawn (Ge4Sb2Te7 ≈ 26 Å)

def draw_d(fig, box):
    x0, y0, w, h = box
    gal = pd.read_csv(GAL / 'gallery.csv').set_index('composition')
    xg = pd.concat([gal.x_GeTe, pd.Series({'Sb2Te3': 0.0, 'GeTe': 1.0})])   # + end members (seed 2)
    mb = mix_best(); mb = mb[mb.composition.isin(D_COMPS)]
    ax = add_ax(fig, x0, y0, w, h - D_TOP - 0.08)
    ax.axhline(0, color=Cd_ZERO, lw=0.5, zorder=0)
    tab = pd.read_csv(SHOW / 'showcase_table_digital_seed0.csv')
    mp = tab[(tab.group == 'gst_series') & (tab.source == 'MP')]
    mp = mp.loc[mp.groupby('x_GeTe').Emix_meV.idxmin()]   # 每个组分只保留能量最低的 MP 结构
    ax.scatter(mp.x_GeTe, mp.Emix_meV, s=8, marker='s', facecolor='none', edgecolor=Cd_MP, lw=0.5, label='MP',
               zorder=2)
    dg = mb[mb['mode'] == 'digital'].groupby('composition').E.min()
    ax.plot(xg[dg.index] - 0.012, dg, 'o', color=Cd_DIG, ms=2.6, mew=0, label='digital', zorder=3)
    a = mb[mb['mode'] == 'tail'].pivot(index='composition', columns='chip', values='E')
    x = xg[a.index] + 0.012
    ax.vlines(x, a.min(1), a.max(1), color=Cd_ANA, lw=0.6, zorder=3)
    ax.plot(x, a.mean(1), 'o', color=Cd_ANA, ms=2.8, mew=0, label='analog', zorder=4)
    ax.text(0.0, -2, 'Sb$_2$Te$_3$', ha='left', va='top', fontsize=FS_ANNOT - 1, color=Cd_END)
    ax.text(1.0, -2, 'GeTe', ha='right', va='top', fontsize=FS_ANNOT - 1, color=Cd_END)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-12, 40); ax.set_yticks([0, 20, 40])
    ax.set_xlabel('GeTe$_x$(Sb$_2$Te$_3$)$_{1-x}$'); ax.set_ylabel('E$_{mix}$ (meV/atom)')
    ax.legend(loc='upper left', fontsize=FS_LEGEND - 1, handletextpad=0.2, borderaxespad=0.1, labelspacing=0.2,
              ncol=3, columnspacing=0.6)
    for j, (el, col) in enumerate(Cd_ATOM.items()):   # 元素图例：三项等间距
        xm = 0.66 + 0.12 * j
        ax.plot(xm, 0.72, 'o', transform=ax.transAxes, ms=3.2, mfc=col, mec=Cd_ATOM_EDGE, mew=0.3, clip_on=False)
        ax.text(xm + 0.03, 0.72, el, transform=ax.transAxes, va='center', fontsize=FS_LEGEND - 1)
    # structures: two rows above the scatter (row-wise in x order), stacking axis horizontal, one common Å scale set by
    # the longer row; each block as wide as its own stacking period
    lab_h, gx, gy = 0.08, 0.10, 0.03
    rows = [D_SHOW[:2], D_SHOW[2:]]
    vfs = {}
    for comp in D_SHOW:
        st = Structure.from_file(GAL / f'{comp}_analog_prim.cif')
        vfs[comp] = (st, view_frame(st.lattice.matrix, st.frac_coords))
    sc = min((w - gx * (len(r) - 1)) / sum(vfs[c][1]['P'] + 0.8 for c in r) for r in rows)   # in per Å
    sh = (D_WIN + 1.6) * sc
    for ri, r in enumerate(rows):
        bx = x0
        by = y0 + h - (ri + 1) * (sh + lab_h) - ri * gy
        for comp in r:
            st, vf = vfs[comp]
            bw_c = (vf['P'] + 0.8) * sc
            g = add_ax(fig, bx, by, bw_c, sh)
            draw_structure(g, st.frac_coords, st.lattice.matrix, [e.symbol for e in st.species], vf, D_WIN, Cd_ATOM,
                           Cd_ATOM_EDGE, Cd_BOND, Cd_GAP, gaps=bool(gal.loc[comp].layered_best_analog),
                           lw_atom=0.25, lw_bond=0.3, horizontal=True)
            g.set_xlim(-0.4, vf['P'] + 0.4); g.set_ylim(-0.8, D_WIN + 0.8)
            g.text(0.0, 1.04, comp_tex(comp), transform=g.transAxes, ha='left', va='bottom', fontsize=FS_ANNOT - 1)
            bx += bw_c + gx
    return ax
panel_run(3, 'd', draw_d)


In [ ]:
# composite
fig, slots = make_layout()
for L, box in zip('abcd', slots):
    first = globals()['draw_' + L](fig, box)
    top = box[1] + box[3] - 0.06              # letters on one baseline just below the canvas top (8 pt glyph fits)
    mark(fig, first, L, dx_in=-0.36, dy_in=top - first.get_position().y1 * H)
export_composite(fig, OUT, 'conf_fig3')
plt.close(fig)
display(NotebookImage(filename=str(OUT / 'conf_fig3.png'), width=int(W * 600 * PREVIEW_SCALE / 2)))
